<a href="https://colab.research.google.com/github/konstanta-asya/Applied_Mashroomatics/blob/main/train_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mushroom classification with CNN

# Налаштування середовища

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!wget http://ptak.felk.cvut.cz/plants/DanishFungiDataset/DF20M-images.tar.gz

--2026-03-16 15:42:57--  http://ptak.felk.cvut.cz/plants/DanishFungiDataset/DF20M-images.tar.gz
Resolving ptak.felk.cvut.cz (ptak.felk.cvut.cz)... 147.32.84.14
Connecting to ptak.felk.cvut.cz (ptak.felk.cvut.cz)|147.32.84.14|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13394117678 (12G) [application/x-gzip]
Saving to: ‘DF20M-images.tar.gz’

DF20M-images.tar.gz 100%[===================>]  12.47G  19.9MB/s    in 11m 19s 

2026-03-16 15:54:17 (18.8 MB/s) - ‘DF20M-images.tar.gz’ saved [13394117678/13394117678]



In [ ]:
!tar -xzf DF20M-images.tar.gz

In [ ]:
import os
print(os.listdir('/content/DF20M'))

['2238569458-35315.JPG', '2238475377-316741.JPG', '2238503500-171581.JPG', '2238585123-38742.JPG', '2856922308-61711.JPG', '2351006924-116737.JPG', '2238581490-186380.JPG', '2238091808-6032.JPG', '2238269907-309001.JPG', '2332527616-264452.JPG', '2238530416-250079.JPG', '2427874584-121872.JPG', '2383043019-117740.JPG', '2425493240-194718.JPG', '2238149054-82033.JPG', '2238498017-320105.JPG', '2238580091-260058.JPG', '2874309419-140295.JPG', '2238529228-27682.JPG', '2238499857-320411.JPG', '2238189609-83699.JPG', '2238478787-317110.JPG', '2238582166-260613.JPG', '2238575114-110891.JPG', '2237963850-153120.JPG', '2382324914-117378.JPG', '2238508370-172478.JPG', '2238499543-22800.JPG', '2237901837-300363.JPG', '2238333365-160331.JPG', '2898620403-68397.JPG', '2964217476-294241.JPG', '2238557971-255461.JPG', '2874311354-140394.JPG', '2237964960-153183.JPG', '2351008042-340271.JPG', '2984615391-295520.JPG', '2421822465-193407.JPG', '2238299092-309159.JPG', '2238469991-240758.JPG', '23830422

In [ ]:
!wget http://ptak.felk.cvut.cz/plants/DanishFungiDataset/DF20M-metadata.zip
!unzip DF20M-metadata.zip

--2026-02-25 08:24:54--  http://ptak.felk.cvut.cz/plants/DanishFungiDataset/DF20M-metadata.zip
Resolving ptak.felk.cvut.cz (ptak.felk.cvut.cz)... 147.32.84.14
Connecting to ptak.felk.cvut.cz (ptak.felk.cvut.cz)|147.32.84.14|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2634951 (2.5M) [application/zip]
Saving to: ‘DF20M-metadata.zip’

DF20M-metadata.zip  100%[===================>]   2.51M  3.40MB/s    in 0.7s    

2026-02-25 08:24:55 (3.40 MB/s) - ‘DF20M-metadata.zip’ saved [2634951/2634951]

Archive:  DF20M-metadata.zip
  inflating: DF20M-train_metadata_PROD.csv  
  inflating: DF20M-public_test_metadata_PROD.csv  


In [ ]:
import os
print(os.listdir('/content'))

['.config', 'DF20M-images.tar.gz', 'DF20M-train_metadata_PROD.csv', 'drive', 'DF20M', 'DF20M-public_test_metadata_PROD.csv', 'DF20M-metadata.zip', 'sample_data']


# Дослідження даних

In [ ]:
import pandas as pd

train_df = pd.read_csv('/content/DF20M-train_metadata_PROD.csv')
print("Розмір:", train_df.shape)
print("Колонки:", train_df.columns.tolist())
print(train_df.head(3))

Розмір: (32753, 33)
Колонки: ['gbifID', 'eventDate', 'year', 'month', 'day', 'countryCode', 'locality', 'taxonID', 'scientificName', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'specificEpithet', 'infraspecificEpithet', 'taxonRank', 'species', 'level0Gid', 'level0Name', 'level1Gid', 'level1Name', 'level2Gid', 'level2Name', 'ImageUniqueID', 'Substrate', 'rightsHolder', 'Latitude', 'Longitude', 'CoorUncert', 'Habitat', 'image_path']
       gbifID            eventDate    year  month   day countryCode  \
0  2862684394  2020-09-17T00:00:00  2020.0    9.0  17.0          DK   
1  2238502117  2017-08-16T00:00:00  2017.0    8.0  16.0          DK   
2  2818074328  2020-07-23T00:00:00  2020.0    7.0  23.0          DK   

               locality  taxonID                    scientificName kingdom  \
0        Langesø, Morud  17215.0  Mycena crocata (Schrad.) P.Kumm.   Fungi   
1                 Virum  10057.0             Agaricus augustus Fr.   Fungi   
2  Gribskov, Enghavehus  20027.0

In [ ]:
import os

print(os.listdir('/content/DF20M')[:5])

sample_path = train_df['image_path'].iloc[0]
print("Приклад шляху:", sample_path)

full_path = f'/content/DF20M/{sample_path}'
print("Файл існує:", os.path.exists(full_path))

['2238569458-35315.JPG', '2238475377-316741.JPG', '2238503500-171581.JPG', '2238585123-38742.JPG', '2856922308-61711.JPG']
Приклад шляху: 2862684394-136762.JPG
Файл існує: True


In [ ]:
!pip install timm -q

# Тренування моделі

In [12]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm

METADATA_TRAIN = '/content/DF20M-train_metadata_PROD.csv'
ROOT_DIR = '/content/DF20M'
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-4
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Використовується: {DEVICE}")

class MushroomDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

        unique_species = self.df['species'].unique()
        self.species_to_id = {sp: idx for idx, sp in enumerate(unique_species)}
        self.df['class_id'] = self.df['species'].map(self.species_to_id)
        self.num_classes = len(unique_species)
        print(f"Кількість класів: {self.num_classes}")
        print(f"Кількість зображень: {len(self.df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['image_path'])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224))
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(int(row['class_id']), dtype=torch.long)
        return image, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

dataset = MushroomDataset(METADATA_TRAIN, ROOT_DIR, transform=train_transform)
NUM_CLASSES = dataset.num_classes

train_loader = DataLoader(dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Accuracy={acc:.2f}%")
    scheduler.step()

    torch.save(model.state_dict(), f'/content/drive/MyDrive/mushroom_model_epoch{epoch+1}.pth')
    print(f"Модель збережена!")

Використовується: cuda
Кількість класів: 180
Кількість зображень: 32753


Epoch 1/10: 100%|██████████| 1024/1024 [19:18<00:00,  1.13s/it]


Epoch 1: Loss=3.3254, Accuracy=27.32%
Модель збережена!


Epoch 2/10: 100%|██████████| 1024/1024 [19:19<00:00,  1.13s/it]


Epoch 2: Loss=1.9647, Accuracy=49.68%
Модель збережена!


...

Epoch 8: Loss=1.0094, Accuracy=73.01%
Модель збережена!


Epoch 9/10: 100%|██████████| 1024/1024 [19:46<00:00,  1.16s/it]


Epoch 9: Loss=1.0051, Accuracy=73.39%
Модель збережена!


Epoch 10/10: 100%|██████████| 1024/1024 [19:27<00:00,  1.14s/it]


Epoch 10: Loss=1.0038, Accuracy=73.23%
Модель збережена!


# Результати

In [13]:
import os

checkpoint_dir = '/content/drive/MyDrive'

if os.path.exists(checkpoint_dir):
    print("📁 Збережені файли:")
    for f in os.listdir(checkpoint_dir):
        if f.startswith('mushroom_model'):
            size = os.path.getsize(os.path.join(checkpoint_dir, f)) / 1e6
            print(f"  - {f} ({size:.1f} MB)")
else:
    print("❌ Папка з чекпоінтами не знайдена")

📁 Збережені файли:
  - mushroom_model_epoch1.pth (17.3 MB)
  - mushroom_model_epoch2.pth (17.3 MB)
  - mushroom_model_epoch3.pth (17.3 MB)
  - mushroom_model_epoch4.pth (17.3 MB)
  - mushroom_model_epoch5.pth (17.3 MB)
  - mushroom_model_epoch6.pth (17.3 MB)
  - mushroom_model_epoch7.pth (17.3 MB)
  - mushroom_model_epoch8.pth (17.3 MB)
  - mushroom_model_epoch9.pth (17.3 MB)
  - mushroom_model_epoch10.pth (17.3 MB)


In [14]:
from google.colab import files
import os

best_model = '/content/drive/MyDrive/mushroom_model_epoch10.pth'

if os.path.exists(best_model):
    files.download(best_model)
    print("✅ Модель завантажено")
else:
    print("❌ Файл не знайдено. Тренування ще не завершено?")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Модель завантажено
